# Derandomized Conformal E-Values

This notebook repeats split-conformal scoring, aggregates conformal e-values, and applies e-BH for stable batch FDR decisions.

## Import

This section loads the dependencies used throughout the notebook.

In [ ]:
import logging

import numpy as np
from oddball import Dataset, load
from pyod.models.iforest import IForest

from nonconform import ConformalDetector, Split
from nonconform.fdr import conformal_e_value_selection
from nonconform.metrics import false_discovery_rate, statistical_power

root_logger = logging.getLogger("nonconform")
if not root_logger.handlers:
    root_logger.addHandler(logging.NullHandler())
root_logger.setLevel(logging.ERROR)

## Setup

Use a fixed Shuttle split, a fixed FDR level, and repeated calibration splits.

In [ ]:
x_train, x_test, y_test = load(Dataset.SHUTTLE, setup=True, seed=1)

alpha = 0.2
n_calib = 1_000
seeds = range(5)

print(f"x_train: {x_train.shape}, x_test: {x_test.shape}")
print(f"y_test positives: {int(y_test.sum())}")
print(f"alpha={alpha}, calibration size={n_calib}")

## Single Split Baseline

`select(...)` is still the default workflow. The e-value path is for repeated split analyses where split instability matters.

In [ ]:
baseline = ConformalDetector(
    detector=IForest(random_state=1),
    strategy=Split(n_calib=n_calib),
    seed=1,
)
baseline.fit(x_train)
baseline_mask = baseline.select(x_test, alpha=alpha)

print(f"Baseline discoveries: {baseline_mask.sum()}")
print(f"Baseline FDR: {false_discovery_rate(y=y_test, y_hat=baseline_mask)}")
print(f"Baseline Power: {statistical_power(y=y_test, y_hat=baseline_mask)}")

## Repeated Split E-Values

Collect `test_scores` and `calib_scores` from each split, then apply `conformal_e_value_selection(...)` once.

In [ ]:
test_scores = []
calib_scores = []

for seed in seeds:
    ce = ConformalDetector(
        detector=IForest(random_state=seed),
        strategy=Split(n_calib=n_calib),
        seed=seed,
    )
    ce.fit(x_train)
    ce.score_samples(x_test)
    result = ce.last_result
    test_scores.append(result.test_scores)
    calib_scores.append(result.calib_scores)

e_result = conformal_e_value_selection(
    np.vstack(test_scores),
    np.vstack(calib_scores),
    alpha=alpha,
)
decisions = e_result.selected

print(f"Derandomized discoveries: {decisions.sum()}")
print(f"Empirical FDR: {false_discovery_rate(y=y_test, y_hat=decisions)}")
print(f"Empirical Power: {statistical_power(y=y_test, y_hat=decisions)}")
print(f"Inner alpha_bh: {e_result.alpha_bh}")